In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [2]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
log_no_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000))
])

log_no_pca.fit(X_train, y_train)
acc_no_pca = accuracy_score(y_test, log_no_pca.predict(X_test))

log_pca_2 = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('model', LogisticRegression(max_iter=5000))
])

log_pca_2.fit(X_train, y_train)
acc_pca_2 = accuracy_score(y_test, log_pca_2.predict(X_test))

log_pca_10 = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('model', LogisticRegression(max_iter=5000))
])

log_pca_10.fit(X_train, y_train)
acc_pca_10 = accuracy_score(y_test, log_pca_10.predict(X_test))

comparison_log = pd.DataFrame({
    'Model': ['Tanpa PCA', 'PCA 2 Komponen', 'PCA 10 Komponen'],
    'Accuracy': [acc_no_pca, acc_pca_2, acc_pca_10]
})

comparison_log

,Model,Accuracy
0,Tanpa PCA,0.973684
1,PCA 2 Komponen,0.991228
2,PCA 10 Komponen,0.982456


In [4]:
tree_no_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier(random_state=42))
])

tree_no_pca.fit(X_train, y_train)
acc_tree_no_pca = accuracy_score(y_test, tree_no_pca.predict(X_test))

tree_pca_10 = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('model', DecisionTreeClassifier(random_state=42))
])

tree_pca_10.fit(X_train, y_train)
acc_tree_pca_10 = accuracy_score(y_test, tree_pca_10.predict(X_test))

comparison_tree = pd.DataFrame({
    'Model': ['Decision Tree Tanpa PCA', 'Decision Tree + PCA 10'],
    'Accuracy': [acc_tree_no_pca, acc_tree_pca_10]
})

comparison_tree

,Model,Accuracy
0,Decision Tree Tanpa PCA,0.947368
1,Decision Tree + PCA 10,0.947368


In [5]:
# sudah ada di Logistic Regression
print("Accuracy PCA 2 Komponen:", acc_pca_2)

Accuracy PCA 2 Komponen: 0.9912280701754386


In [6]:
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('model', KNeighborsClassifier())
])

param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11],
    'model__weights': ['uniform', 'distance']
}

grid = GridSearchCV(knn_pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)
print("Test Accuracy:", grid.score(X_test, y_test))

Best Params: {'model__n_neighbors': 5, 'model__weights': 'uniform'}
Best CV Score: 0.9604395604395606
Test Accuracy: 0.956140350877193


In [7]:
X_scaled = StandardScaler().fit_transform(X)

pca_full = PCA()
pca_full.fit(X_scaled)

variance_table = pd.DataFrame({
    'Komponen': ['PC1', 'PC2', 'PC3'],
    'Explained Variance': pca_full.explained_variance_ratio_[:3]
})

variance_table

,Komponen,Explained Variance
0,PC1,0.442720
1,PC2,0.189712
2,PC3,0.093932


In [8]:
pca_80 = PCA(n_components=0.80)
pca_80.fit(X_scaled)

pca_90 = PCA(n_components=0.90)
pca_90.fit(X_scaled)

pca_95 = PCA(n_components=0.95)
pca_95.fit(X_scaled)

print("Komponen 80% :", pca_80.n_components_)
print("Komponen 90% :", pca_90.n_components_)
print("Komponen 95% :", pca_95.n_components_)

Komponen 80% : 5
Komponen 90% : 7
Komponen 95% : 10
